In [1]:
import pandas as pd
import numpy as np
from scipy.ndimage import gaussian_filter1d
from sklearn.metrics import f1_score
import matplotlib.pyplot as plt
from sklearn.metrics import f1_score, classification_report

# 1. LE MOTEUR : L'algorithme de Viterbi en Log-espace

from Data.Tool import *
df = pd.read_csv("Resultat/predictions_sur_le_train-2.csv")
df_train = load_and_prepare(
        corpus_path = "Data/train/corpus.tache1.learn.utf8",
        pkl_path    = "Data/train/sequences_fasttext_fr.pkl",
        vector_size = 300,
    )
y_true = df['Vrai_Label_ID']
prob_class_0 = df['Prob_Mitterrand'] # Ta probabilité pour la classe 0

# 2. Définir la métrique pondérée (Poids 6.6 sur la classe 0)
def weighted_f1(y_vrai, y_pred):
    f1_0 = f1_score(y_vrai, y_pred, pos_label=0, zero_division=0)
    f1_1 = f1_score(y_vrai, y_pred, pos_label=1, zero_division=0)
    return (6.6 * f1_0 + 1.0 * f1_1) / 7.6


📂 Chargement du corpus...
   → 57413 lignes chargées
🧹 Nettoyage du texte...
📦 Chargement des séquences depuis 'Data/train/sequences_fasttext_fr.pkl'...
   → 57413 séquences fusionnées
🗑️  Suppression des lignes vides...
   → 35 lignes supprimées | 57378 lignes restantes
🔍 Vérification cohérence tokens ↔ vecteurs...


Vérification: 100%|██████████| 57378/57378 [00:01<00:00, 37094.98it/s]


✅ Lignes finales       : 57378
✅ Balance C/M         : {'C': 49855, 'M': 7523}
✅ Erreurs restantes  : 0
🎉 Pipeline prêt pour l'entraînement !


In [2]:
structure = df_train[['Texte', 'Doc_ID', 'Sentence_ID', 'Label']].drop_duplicates(subset=['Texte'])

df_final = pd.merge(df, structure, on='Texte', how='inner')
df_final = df_final.sort_values(by=['Doc_ID', 'Sentence_ID']).reset_index(drop=True)

print(f"Nombre de lignes après fusion : {len(df_final)}")
print(f"Nombre de Doc_ID uniques : {df_final['Doc_ID'].nunique()}")
# Vérifier si on a toujours nos blocs de ~3 phrases
df_final['is_cont'] = (df_final['Doc_ID'] == df_final['Doc_ID'].shift()) & \
                      (df_final['Sentence_ID'] == df_final['Sentence_ID'].shift() + 1)

continuite = df_final['is_cont'].mean() * 100
print(f"Score de continuité du dataset fusionné : {continuite:.2f}%")

Nombre de lignes après fusion : 45902
Nombre de Doc_ID uniques : 585
Score de continuité du dataset fusionné : 77.84%


In [3]:
def compute_real_transitions(df):
    # On ne compte la transition que si Sentence_ID suit n+1 dans le même Doc
    mask_cont = (df['Doc_ID'] == df['Doc_ID'].shift(-1)) & \
                (df['Sentence_ID'] + 1 == df['Sentence_ID'].shift(-1))
    
    transitions = np.zeros((2, 2))
    
    # On itère sur les lignes qui ont une suite continue
    valid_indices = df.index[mask_cont]
    for idx in valid_indices:
        current_label = df.at[idx, 'Label']
        next_label = df.at[idx+1, 'Label']
        # Conversion si tes labels sont 'M'/'C' (adapte selon tes données)
        i = 0 if current_label == 'M' else 1
        j = 0 if next_label == 'M' else 1
        transitions[i, j] += 1
        
    A = transitions / transitions.sum(axis=1)[:, np.newaxis]
    return A

A_real = compute_real_transitions(df_final)
print("Matrice de Transition Réelle (A) :\n", A_real)

Matrice de Transition Réelle (A) :
 [[0.94879267 0.05120733]
 [0.00792162 0.99207838]]


In [4]:
def viterbi_hmm(probs_c0, trans_matrix, pi, weight_c0=3):
    n = len(probs_c0)
    
    # Passage en log pour la stabilité (évite les nombres trop petits)
    log_pi = np.log(pi + 1e-12)
    log_trans = np.log(trans_matrix + 1e-12)
    
    # Émissions : on booste la classe 0 avec ton facteur 6.6
    # log(p * weight) = log(p) + log(weight)
    emissions = np.vstack([probs_c0, 1 - probs_c0])
    log_emissions = np.log(emissions + 1e-12)
    log_emissions[0, :] += np.log(weight_c0) 

    viterbi_table = np.zeros((2, n))
    backpointer = np.zeros((2, n), dtype=int)

    # Initialisation (t=0)
    viterbi_table[:, 0] = log_pi + log_emissions[:, 0]

    # Forward pass
    for t in range(1, n):
        for s in range(2):
            # On cherche le max de (score précédent + transition + émission actuelle)
            scores = viterbi_table[:, t-1] + log_trans[:, s] + log_emissions[s, t]
            viterbi_table[s, t] = np.max(scores)
            backpointer[s, t] = np.argmax(scores)

    # Backward pass (Backtracking)
    path = np.zeros(n, dtype=int)
    path[n-1] = np.argmax(viterbi_table[:, n-1])
    for t in range(n-2, -1, -1):
        path[t] = backpointer[path[t+1], t+1]
        
    return path

def viterbi_hmm(probs_c0, trans_matrix, pi, weight_c0=6.6):
    n = len(probs_c0)
    
    # Passage en log pour la stabilité (évite les nombres trop petits)
    log_pi = np.log(pi + 1e-12)
    log_trans = np.log(trans_matrix + 1e-12)
    
    # Émissions : on booste la classe 0 avec ton facteur 6.6
    # log(p * weight) = log(p) + log(weight)
    emissions = np.vstack([probs_c0, 1 - probs_c0])
    log_emissions = np.log(emissions + 1e-12)
    log_emissions[0, :] += np.log(weight_c0) 

    viterbi_table = np.zeros((2, n))
    backpointer = np.zeros((2, n), dtype=int)

    # Initialisation (t=0)
    viterbi_table[:, 0] = log_pi + log_emissions[:, 0]

    # Forward pass
    for t in range(1, n):
        for s in range(2):
            # On cherche le max de (score précédent + transition + émission actuelle)
            scores = viterbi_table[:, t-1] + log_trans[:, s] + log_emissions[s, t]
            viterbi_table[s, t] = np.max(scores)
            backpointer[s, t] = np.argmax(scores)

    # Backward pass (Backtracking)
    path = np.zeros(n, dtype=int)
    path[n-1] = np.argmax(viterbi_table[:, n-1])
    for t in range(n-2, -1, -1):
        path[t] = backpointer[path[t+1], t+1]
        
    return path

# 2. L'ORCHESTRE : Application par segments Doc_ID
def apply_viterbi_segmented(df, matrix_A, weight_c0=6.6):
    # On identifie les segments continus pour ne pas lisser entre deux débats
    df['is_break'] = (df['Doc_ID'] != df['Doc_ID'].shift()) | \
                     (df['Sentence_ID'] != df['Sentence_ID'].shift() + 1)
    df['segment_id'] = df['is_break'].cumsum()
    
    final_labels = []
    pi_global = np.array([0.13, 0.87]) # Proba de départ moyenne

    for _, segment in df.groupby('segment_id'):
        probs = segment['Prob_Mitterrand'].values
        
        if len(segment) < 2:
            # Si phrase isolée, on applique juste le seuil pondéré
            seuil = 1 / (weight_c0 + 1)
            preds = (probs > seuil).astype(int)
        else:
            # Application du HMM sur le segment continu
            preds = viterbi_hmm(probs, matrix_A, pi_global, weight_c0)
            
        final_labels.extend(preds)
    
    return np.array(final_labels)

def forward_backward_smoother(probs_c0, trans_matrix, pi, weight_c0=6.6):
    n = len(probs_c0)
    # On prépare les émissions (avec ton poids 6.6)
    # Attention : on reste en espace linéaire (pas log) pour les sommes, 
    # mais on utilise un facteur d'échelle pour éviter l'underflow.
    obs = np.vstack([probs_c0 * weight_c0, 1 - probs_c0])
    obs /= obs.sum(axis=0) # Normalisation locale des émissions

    # Forward Pass (Alpha)
    alpha = np.zeros((2, n))
    scale = np.zeros(n)
    
    alpha[:, 0] = pi * obs[:, 0]
    scale[0] = alpha[:, 0].sum()
    alpha[:, 0] /= scale[0]
    
    for t in range(1, n):
        alpha[:, t] = (alpha[:, t-1] @ trans_matrix) * obs[:, t]
        scale[t] = alpha[:, t].sum()
        alpha[:, t] /= scale[t]

    # Backward Pass (Beta)
    beta = np.zeros((2, n))
    beta[:, n-1] = 1.0
    
    for t in range(n-2, -1, -1):
        beta[:, t] = (trans_matrix @ (obs[:, t+1] * beta[:, t+1]))
        beta[:, t] /= scale[t+1] # On utilise le même scale pour la stabilité

    # Calcul des probabilités postérieures (Gamma)
    posterior = alpha * beta
    posterior /= posterior.sum(axis=0)
    
    return posterior[0, :] # On renvoie uniquement la proba lissée de la classe 0

# --- Wrapper pour les segments ---
def apply_hmm_proba_segmented(df, matrix_A, weight_c0=6.6):
    df['is_break'] = (df['Doc_ID'] != df['Doc_ID'].shift()) | \
                     (df['Sentence_ID'] != df['Sentence_ID'].shift() + 1)
    df['segment_id'] = df['is_break'].cumsum()
    
    final_probs = []
    pi_global = np.array([0.13, 0.87])

    for _, segment in df.groupby('segment_id'):
        p_raw = segment['Prob_Mitterrand'].values
        
        if len(p_raw) < 2:
            # Pour une phrase seule, on applique juste le boost du poids
            p_boosted = (p_raw * weight_c0) / (p_raw * weight_c0 + (1 - p_raw))
            final_probs.extend(p_boosted)
        else:
            # Lissage Forward-Backward
            p_smooth = forward_backward_smoother(p_raw, matrix_A, pi_global, weight_c0)
            final_probs.extend(p_smooth)
            
    return np.array(final_probs)

In [6]:
from sklearn.model_selection import GroupKFold

# 1. Préparation des données
# On s'assure que les labels sont numériques pour le calcul
df_final['target'] = df_final['Label'].map({'M': 0, 'C': 1})
X = df_final 
y = df_final['target']
groups = df_final['Doc_ID']

gkf = GroupKFold(n_splits=5)

scores_f1_c0 = []
fold = 1

print(f"--- Début du GroupKFold (5 splits) ---")

for train_idx, val_idx in gkf.split(X, y, groups):
    # Split Train / Validation
    df_train_fold = X.iloc[train_idx].copy()
    df_val_fold = X.iloc[val_idx].copy()
    
    # A. "Entraînement" du HMM sur le Train Fold (Calcul de la matrice A)
    # On utilise la fonction de calcul de matrice définie précédemment
    A_fold = compute_real_transitions(df_train_fold)
    
    # B. Inférence Viterbi sur le Validation Fold
    # On applique le HMM avec ton poids de 6.6
    preds_val = apply_viterbi_segmented(df_val_fold, A_fold, weight_c0=6.6)
    
    # C. Calcul du score pour ce fold (Classe 0)
    y_val_true = df_val_fold['target'].values
    score = f1_score(y_val_true, preds_val, pos_label=0)
    scores_f1_c0.append(score)
    
    print(f"Fold {fold} | F1 Classe 0: {score:.4f} | Matrice A[0,0]: {A_fold[0,0]:.3f}")
    fold += 1

# 2. Résultats finaux
print("\n--- RÉSULTATS FINAUX ---")
print(f"F1-score moyen (Classe 0) : {np.mean(scores_f1_c0):.4f} (+/- {np.std(scores_f1_c0):.4f})")

--- Début du GroupKFold (5 splits) ---
Fold 1 | F1 Classe 0: 0.8485 | Matrice A[0,0]: 0.948
Fold 2 | F1 Classe 0: 0.8014 | Matrice A[0,0]: 0.948
Fold 3 | F1 Classe 0: 0.8369 | Matrice A[0,0]: 0.949
Fold 4 | F1 Classe 0: 0.8359 | Matrice A[0,0]: 0.948
Fold 5 | F1 Classe 0: 0.8245 | Matrice A[0,0]: 0.951

--- RÉSULTATS FINAUX ---
F1-score moyen (Classe 0) : 0.8294 (+/- 0.0160)


In [7]:
df_test_raw = load_tagged_test("Data/test/corpus.tache1.test.utf8")
df_probs = pd.read_csv("Resultat/submission-pres-4.csv", header=None, names=['Prob_Mitterrand'])


assert (len(df_test_raw) == len(df_probs))
df_test_full = pd.concat([df_test_raw, df_probs], axis=1)

df_sorted = df_test_full.sort_values(['Doc_ID', 'Sentence_ID']).copy()
df_sorted.head()

,original_index,Doc_ID,Sentence_ID,Texte,Prob_Mitterrand
0,0,105,1,"En répondant à votre invitation, en effectuant...",0.001728
1,1,105,2,"Et ce moment exceptionnel, j'ai essayé de le c...",0.001532
2,2,105,3,Et le musée.,0.362105
3,3,105,4,Je parlais en même temps que le Maire d'Angers...,0.892779
4,4,105,5,<nom> devait plus tard n'y être pas pour rien ...,0.985122


In [9]:
df_test_full = pd.concat([df_test_raw, df_probs], axis=1)

# 2. Trier pour le contexte chronologique
df_sorted = df_test_full.sort_values(['Doc_ID', 'Sentence_ID']).copy()

# 3. Calculer les probabilités lissées
probs_smooth = apply_hmm_proba_segmented(df_sorted, A_real, weight_c0=6.6)
df_sorted['Prob_HMM'] = probs_smooth

# 4. Remettre dans l'ordre original du fichier de test
df_final = df_sorted.sort_values('original_index')

# 5. Sauvegarde des probabilités (format float)
df_final['Prob_HMM'].to_csv("submission-pres-5.csv", index=False, header=False)
df_final.head()

,original_index,Doc_ID,Sentence_ID,Texte,Prob_Mitterrand,is_break,segment_id,Prob_HMM
0,0,105,1,"En répondant à votre invitation, en effectuant...",0.001728,True,1,0.001603
1,1,105,2,"Et ce moment exceptionnel, j'ai essayé de le c...",0.001532,False,1,0.008929
2,2,105,3,Et le musée.,0.362105,False,1,0.774328
3,3,105,4,Je parlais en même temps que le Maire d'Angers...,0.892779,False,1,0.987921
4,4,105,5,<nom> devait plus tard n'y être pas pour rien ...,0.985122,False,1,0.991150
